# Notebook 10B: scaled pretraining, from `chemprop_chemeleon`

Our multitask network saw 33,000 molecules over 20 assay columns and gave the single
largest win of this project, +0.0118 development Spearman. The published methods above us
pretrain every backbone on public bioactivity at far greater scale. This is the same idea
with a corpus about five times larger and thirty times wider.

| source | compounds | tasks |
| :--- | ---: | ---: |
| ToxCast | 8,597 | 617 |
| MUV | 93,000 | 17 |
| HIV | 41,000 | 1 |
| Tox21 | 7,831 | 12 |
| PubChem AID 1851 | 16,559 | 5 |
| ChEMBL CYP | 13,841 | 7 |
| the challenge itself | 4,905 | 8 |

Two stages, which is faster than one big table: the encoder is pretrained once on
everything, then fine-tuned per fold on the challenge labels alone. Roughly 15 minutes
then 7 minutes, against 4.5 hours for a single combined run.

**Run the smoke cell first.** It builds the corpus, trains fifty molecules for two epochs
and prints every shape. Two minutes. If anything is wrong you see it there rather than
ninety minutes in. Every corpus source is wrapped so a failed download prints a warning
and is skipped instead of killing the run.

The model and training code is reused unchanged from notebook 8, which ran clean.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or 'NO GPU')

In [ ]:
# Labels, folds and holdout mask only. The encoder is trained here, so no cached
# embeddings are needed and this payload is 1 MB rather than 8.6.
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/tiny_6387286b00.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
OUTDIR = '/content/results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/cyp_results'
except Exception as e:
    print('Drive unavailable:', type(e).__name__, '- results live on the runtime disk')
os.makedirs(OUTDIR, exist_ok=True)
print('writing to', OUTDIR)

In [ ]:
!pip -q install chemprop==2.3.1

In [ ]:
import gc, json, time, pickle, urllib.request
import numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

CACHE=Path('/content/cache'); OUT=Path(OUTDIR)
ISO=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
CKNAME='chemprop_chemeleon.pt'; TAG='B'

meta=pd.read_csv(CACHE/'meta_train.csv.gz'); mete=pd.read_csv(CACHE/'meta_test.csv.gz')
hold=np.load(CACHE/'holdout_mask.npy')
G=np.load(CACHE/'scaffold_groups.npy', allow_pickle=True)
AUX=np.load(CACHE/'aux_train.npy'); QH=np.load(CACHE/'qhts_tr.npy')
SM_TR=meta['SMILES'].to_numpy(); SM_TE=mete['SMILES'].to_numpy()

CK_PATH=f'/content/{CKNAME}'
if not Path(CK_PATH).exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/jeremycheminf/'
        f'openadmet_scripts/main/CYP_Challenge/checkpoints/{CKNAME}', CK_PATH)
print('checkpoint', CKNAME, round(Path(CK_PATH).stat().st_size/1e6,1), 'MB')
print('device', DEVICE, '| challenge rows', len(SM_TR), '| holdout', int(hold.sum()))

## Corpus. Every source is optional; a failure is skipped, not fatal.

In [ ]:
S3='https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/'
def canon(s):
    m=Chem.MolFromSmiles(s) if isinstance(s,str) else None
    return Chem.MolToSmiles(m) if m is not None else None

def molnet(fname):
    """MoleculeNet CSVs: one SMILES column plus every numeric column as a task."""
    p=f'/content/{fname}'
    if not Path(p).exists(): urllib.request.urlretrieve(S3+fname, p)
    d=pd.read_csv(p)
    sc=next(c for c in d.columns if c.lower()=='smiles')
    tasks=[c for c in d.columns if c!=sc and pd.api.types.is_numeric_dtype(d[c])]
    return d[[sc]+tasks].rename(columns={sc:'smiles'})

def pubchem_1851():
    p='/content/aid1851.csv'
    if not Path(p).exists():
        urllib.request.urlretrieve('https://pubchem.ncbi.nlm.nih.gov/assay/pcget.cgi?'
            'query=download&record_type=datatable&actvty=all&response_type=save&aid=1851', p)
    r=pd.read_csv(p, low_memory=False, skiprows=[1])
    r['v']=-pd.to_numeric(r['Fit_LogAC50'], errors='coerce')
    r['smiles']=r['PUBCHEM_EXT_DATASOURCE_SMILES']
    r=r.dropna(subset=['smiles','v','Panel Name'])
    w=r.pivot_table(index='smiles', columns='Panel Name', values='v', aggfunc='median')
    return w.reset_index()

def chembl_cyp():
    p='/content/cyp_chembl.csv'
    if not Path(p).exists():
        urllib.request.urlretrieve('https://raw.githubusercontent.com/jeremycheminf/'
            'openadmet_scripts/main/CYP_Challenge/data/external/cyp_chembl.csv', p)
    d=pd.read_csv(p)
    keep=[c for c in d.columns if c.endswith('_PIC50') and d[c].notna().sum()>=200]
    return d[['smiles']+keep]

SOURCES=[('toxcast', lambda: molnet('toxcast_data.csv.gz')),
         ('muv',     lambda: molnet('muv.csv.gz')),
         ('hiv',     lambda: molnet('HIV.csv')),
         ('tox21',   lambda: molnet('tox21.csv.gz')),
         ('qhts',    pubchem_1851),
         ('chembl',  chembl_cyp)]

frames=[]
for name, fn in SOURCES:
    try:
        d=fn()
        d=d.dropna(subset=['smiles'])
        d.columns=['smiles']+[f'{name}_{c}' for c in d.columns[1:]]
        frames.append(d)
        print(f'  {name:8s} {len(d):7d} compounds, {d.shape[1]-1:4d} tasks', flush=True)
    except Exception as e:
        print(f'  {name:8s} SKIPPED ({type(e).__name__}: {str(e)[:70]})', flush=True)
assert frames, 'no corpus source loaded; stop here'
print(f'\n{len(frames)} of {len(SOURCES)} sources loaded')

In [ ]:
t0=time.time()
ours={c for c in map(canon, np.concatenate([SM_TR, SM_TE])) if c}
parts=[]
for d in frames:
    d=d.copy(); d['canon']=[canon(s) for s in d['smiles']]
    d=d.dropna(subset=['canon'])
    d=d[~d['canon'].isin(ours)]                      # nothing the blind set contains
    d=d.drop(columns=['smiles']).groupby('canon').median()
    parts.append(d)
corpus=pd.concat(parts, axis=1)
print(f'corpus {corpus.shape[0]} compounds, {corpus.shape[1]} tasks '
      f'({time.time()-t0:.0f}s)')

CH=[f'chal_{i}' for i in ISO]+[f'scr_{i}' for i in ISO]
HEADS=CH+list(corpus.columns)
n_ch=len(SM_TR)
SM_ALL=np.concatenate([SM_TR, corpus.index.to_numpy()])
Y=np.full((len(SM_ALL), len(HEADS)), np.nan, np.float32)
for j,i in enumerate(ISO):
    Y[:n_ch, j]=meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float)
    Y[:n_ch, 4+j]=AUX[:, j]
Y[n_ch:, len(CH):]=corpus.to_numpy(float)
M=np.isfinite(Y)
_n=Y.copy(); _n[np.flatnonzero(hold)]=np.nan          # never normalise on held-out labels
mu=np.nanmean(_n,0); sd=np.nanstd(_n,0)
sd=np.where(np.isfinite(sd)&(sd>0), sd, 1.0); mu=np.nan_to_num(mu)
Yz_nan=((Y-mu)/sd).astype(np.float32)
print(f'table {Y.shape} | labels {int(M.sum()):,} | challenge rows 0..{n_ch-1}')
print(f'tasks with at least 50 labels: {int((M.sum(0)>=50).sum())} of {len(HEADS)}')

In [ ]:
from chemprop import data, featurizers
from chemprop.models.model import MPNN

FZ = featurizers.SimpleMoleculeMolGraphFeaturizer()

# Targets live inside the datapoints, so chemprop's own loader batches them with the
# graphs and nothing has to be re-featurised per minibatch. NaN marks "not measured"
# and becomes the loss mask.
PTS_ALL = [data.MoleculeDatapoint.from_smi(s, Yz_nan[i])
           for i, s in enumerate(SM_ALL)]
PTS_TE = [data.MoleculeDatapoint.from_smi(s, np.zeros(len(HEADS), np.float32))
          for s in SM_TE]
DS_TE = data.MoleculeDataset(PTS_TE, FZ)

def loader(idx=None, bs=128, shuffle=False, test=False):
    pts = PTS_TE if test else ([PTS_ALL[i] for i in idx] if idx is not None else PTS_ALL)
    return data.build_dataloader(data.MoleculeDataset(pts, FZ), num_workers=0,
                                 batch_size=bs, shuffle=shuffle, drop_last=False)

def _fp(m, bmg, V_d, X_d):
    # BatchMolGraph.to() moves in place and returns None, unlike a torch tensor, so it
    # must be called as a statement rather than used as an expression.
    if hasattr(bmg, 'to'):
        bmg.to(DEVICE)
    V_d = V_d.to(DEVICE) if torch.is_tensor(V_d) else V_d
    X_d = X_d.to(DEVICE) if torch.is_tensor(X_d) else X_d
    return m.fingerprint(bmg, V_d, X_d)

class Multi(nn.Module):
    def __init__(self, ckpt, n_out, warm=True):
        super().__init__()
        m = MPNN.load_from_file(ckpt)
        if not warm:
            for p in m.parameters():
                nn.init.xavier_uniform_(p) if p.dim() > 1 else nn.init.zeros_(p)
        self.enc = m
        with torch.no_grad():                      # probe on CPU, before .to(DEVICE)
            bmg, V_d, X_d, *_ = next(iter(loader([0, 1, 2, 3], bs=4)))
            d = m.fingerprint(bmg, V_d, X_d).shape[1]
        self.head = nn.Sequential(nn.Linear(d, 512), nn.ReLU(), nn.Dropout(0.1),
                                  nn.Linear(512, n_out))

    def forward(self, bmg, V_d, X_d):
        return self.head(_fp(self.enc, bmg, V_d, X_d))

def train(tr_idx, epochs=25, warm=True, lr=3e-4, bs=128, seed=0, init_state=None):
    torch.manual_seed(seed)
    m = Multi(CK_PATH, len(HEADS), warm)
    if init_state is not None:
        m.load_state_dict(torch.load(init_state, map_location='cpu'))
    m = m.to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-5)
    ld = loader(tr_idx, bs=bs, shuffle=True)
    for ep in range(epochs):
        m.train()
        for batch in ld:
            bmg, V_d, X_d, Yb, *_ = batch
            t = Yb.to(DEVICE).float()
            msk = torch.isfinite(t)
            if msk.sum() == 0:
                continue
            opt.zero_grad()
            p = m(bmg, V_d, X_d)
            loss = (((p - torch.nan_to_num(t)) ** 2) * msk).sum() / msk.sum()
            loss.backward()
            opt.step()
    return m

@torch.no_grad()
def predict(m, idx=None, bs=256, test=False):
    m.eval()
    out = [m(*batch[:3]).cpu().numpy() for batch in loader(idx, bs=bs, test=test)]
    return np.vstack(out) * sd + mu

## Smoke test, about two minutes

Builds fifty molecules, trains two epochs, prints every shape. Nothing below runs long
enough to waste your time if something here is wrong.

In [ ]:
idx=np.concatenate([np.arange(30), np.arange(n_ch, n_ch+20)])
print('batch shapes:')
bmg, V_d, X_d, Yb, *_ = next(iter(loader(idx, bs=8)))
print('  targets', tuple(Yb.shape), '| finite', int(torch.isfinite(Yb).sum()))
t0=time.time(); m=train(idx, epochs=2); print(f'  trained 2 epochs ({time.time()-t0:.0f}s)')
p=predict(m, idx[:10]); print('  predictions', p.shape, '| finite', bool(np.isfinite(p).all()))
assert p.shape[1]==len(HEADS), 'head count mismatch'
del m; gc.collect(); torch.cuda.empty_cache()
print('\nSMOKE TEST PASSED - safe to run the cells below')

## Stage 1, pretrain the encoder on the corpus

Challenge rows are excluded here, so nothing from the holdout reaches the encoder before
fine-tuning. Checkpointed, so a disconnect costs this stage once.

In [ ]:
PRE_EPOCHS = 8
FT_EPOCHS  = 15
STATE = OUT / f'nb10{TAG}_pretrained.pt'

corpus_idx = np.arange(n_ch, len(SM_ALL))
if STATE.exists():
    print('pretrained encoder already on disk, skipping stage 1')
else:
    t0=time.time()
    m = train(corpus_idx, epochs=PRE_EPOCHS)
    torch.save(m.state_dict(), STATE)
    print(f'stage 1 done, {len(corpus_idx)} molecules, {PRE_EPOCHS} epochs '
          f'({time.time()-t0:.0f}s)')
    del m; gc.collect(); torch.cuda.empty_cache()

## Stage 2, fine-tune per fold on the challenge labels

In [ ]:
y_all={i: meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float) for i in ISO}
obs=np.isfinite(np.column_stack([y_all[i] for i in ISO])).any(1)
dev_rows=np.flatnonzero(obs & ~hold); ho_rows=np.flatnonzero(obs & hold)
folds=list(GroupKFold(5).split(dev_rows, dev_rows, G[dev_rows]))

# Corpus rows anchor the encoder during fine-tuning, but all 150,000 of them would make
# the challenge labels 2% of the gradient and each fit half an hour. 12,000 keeps the
# roughly 22% challenge share that the 33,000-molecule version had, at 2.5 min a fit.
CORPUS_FT = 12000
rng = np.random.default_rng(0)

CKPT = OUT / f'nb10{TAG}j_state.pkl'
acc, done = ({}, set())
if CKPT.exists():
    st=pickle.load(open(CKPT,'rb')); acc=st['acc']; done=st['done']
    print('resuming;', len(done), 'fits complete')
def save(): pickle.dump({'acc':acc,'done':done}, open(CKPT,'wb'))

t0=time.time()
for f,(tr,va) in enumerate(folds):
    k=f'fold_{f}'
    if k in done: continue
    # Challenge rows alone drag the encoder off its pretraining distribution, which is
    # the failure mode recorded for CheMeleon, MoLFormer and ChemBERTa. A corpus sample
    # holds it in place without swamping the task.
    cft = rng.choice(corpus_idx, min(CORPUS_FT, len(corpus_idx)), replace=False)
    m=train(np.concatenate([dev_rows[tr], cft]), epochs=FT_EPOCHS, init_state=STATE)
    acc[k]=predict(m, dev_rows[va]); done.add(k); save()
    print(f'  fold {f} ({time.time()-t0:.0f}s)', flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()
for k, idx, kw in [('hold', ho_rows, {}), ('test', None, {'test': True})]:
    if k in done: continue
    cft = rng.choice(corpus_idx, min(CORPUS_FT, len(corpus_idx)), replace=False)
    rows = (np.concatenate([dev_rows, cft]) if k=='hold'
            else np.concatenate([dev_rows, ho_rows, cft]))
    m=train(rows, epochs=FT_EPOCHS, init_state=STATE)
    acc[k]=predict(m, idx, **kw); done.add(k); save()
    print(f'  {k} model ({time.time()-t0:.0f}s)', flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()

oof=np.full((n_ch, len(HEADS)), np.nan, np.float32)
for f,(tr,va) in enumerate(folds): oof[dev_rows[va]]=acc[f'fold_{f}']
hp, tp = acc['hold'], acc['test']
RESULTS=[]
for j,i in enumerate(ISO):
    y=y_all[i]
    d=dev_rows[np.isfinite(y[dev_rows])]; h=ho_rows[np.isfinite(y[ho_rows])]
    np.savez_compressed(OUT/f'bigj{TAG}_{i}.npz', oof=oof[d,j], dev=d,
                        hold=hp[np.isin(ho_rows,h),j], ho=h, test=tp[:,j])
    r=float(spearmanr(y[d],oof[d,j]).statistic)
    rh=float(spearmanr(y[h],hp[np.isin(ho_rows,h),j]).statistic)
    RESULTS.append(dict(iso=i, dev_spearman=r, holdout_spearman=rh))
    json.dump(RESULTS, open(OUT/f'results_nb10{TAG}j.json','w'), indent=2)
    print(f'{i}: dev rho {r:.4f} | holdout rho {rh:.4f}', flush=True)
print('\n33k-molecule version:  dev 0.5577 0.6935 0.4515 0.8022')
print('                      holdout 0.5759 0.6877 0.5529 0.8443')
print('If this is not clearly better, the scaling premise is wrong and we stop.')

import shutil
from google.colab import files
shutil.make_archive(f'/content/nb10{TAG}j_results','zip',OUT)
files.download(f'/content/nb10{TAG}j_results.zip')